In [0]:
import csv
from pathlib import Path

# Espacio exclusivo para las pruebas.
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.taco_bueno_pruebas
""")

spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.taco_bueno_pruebas.entradas
""")

ruta = Path(
    "/Volumes/workspace/taco_bueno_pruebas/"
    "entradas/casos_silver_v1.csv"
)

# Registro base con campos válidos.
base = {
    "store_number": "PRUEBA",
    "date": "3/5/2025",
    "total": "10.00",
    "check_number": "",
    "time_card": "TEST",
    "begin_time": "10:00 AM",
    "end_time": "10:05 AM",
    "total_discount": "0",
    "order_voided": "0",
    "tax": "0",
    "quantity": "1",
    "transaction_type": "Sale",
    "menu_item": "PRODUCTO DE PRUEBA",
    "menu_item_ext_ref": "",
    "pop": "",
    "pos_code": "TEST",
    "menu_price": "10.00",
    "item_discount": "0",
    "item_voided": "0"
}

# Cada caso cambia un campo del registro base.
casos = [
    ("valido_fecha_americana", {}),
    ("valido_fecha_iso", {"date": "2025-03-05"}),
    ("tienda_vacia", {"store_number": ""}),
    ("fecha_invalida", {"date": "2/30/2025"}),
    ("cantidad_no_numerica", {"quantity": "abc"}),
    ("precio_faltante", {"menu_price": ""}),
    ("hora_inicio_invalida", {"begin_time": "25:99"}),
    ("hora_fin_invalida", {"end_time": "25:99"})
]

registros = []

for numero, (nombre, cambios) in enumerate(casos, start=1):
    registro = base.copy()
    registro.update(cambios)
    registro["check_number"] = f"TEST_{numero}"
    registro["caso_prueba"] = nombre
    registros.append(registro)

# Conservar el archivo si repetimos esta celda.
if not ruta.exists():
    with ruta.open("w", encoding="utf-8", newline="") as archivo:
        escritor = csv.DictWriter(
            archivo,
            fieldnames=list(registros[0].keys())
        )
        escritor.writeheader()
        escritor.writerows(registros)

    print("Archivo de prueba creado.")
else:
    print("El archivo de prueba ya existe y se conserva.")

# Mostrar lo que quedó guardado.
datos_prueba = (
    spark.read
    .option("header", "true")
    .csv(str(ruta))
)

print("Registros de prueba:", datos_prueba.count())
display(datos_prueba)

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.taco_bueno_pruebas
""")

display(spark.sql("""
    SHOW SCHEMAS IN workspace LIKE 'taco_bueno_pruebas'
"""))

In [0]:
resultado = spark.table(
    "workspace.taco_bueno_pruebas.silver_pos_lines"
)

filas = resultado.count()

print("Filas conservadas:", filas)

assert filas == 8, (
    f"Esperábamos conservar ocho registros, pero hay {filas}."
)

display(
    resultado.select(
        "caso_prueba",
        "store_number",
        "fecha",
        "quantity_n",
        "error_conversion_numerica",
        "menu_price_n",
        "hora_inicio",
        "hora_fin"
    ).orderBy("caso_prueba")
)

In [0]:
from datetime import date, time
from decimal import Decimal

# Resultado esperado para un registro válido.
base_esperada = {
    "store_number": "PRUEBA",
    "fecha": date(2025, 3, 5),
    "quantity_n": Decimal("1"),
    "menu_price_n": Decimal("10"),
    "error_conversion_numerica": False,
    "importe_linea": Decimal("10"),
    "hora_inicio": time(10, 0),
    "hora_fin": time(10, 5)
}

# Cambios esperados para cada error introducido.
casos_esperados = {
    "valido_fecha_americana": {},
    "valido_fecha_iso": {},
    "tienda_vacia": {
        "store_number": None
    },
    "fecha_invalida": {
        "fecha": None
    },
    "cantidad_no_numerica": {
        "quantity_n": None,
        "error_conversion_numerica": True,
        "importe_linea": None
    },
    "precio_faltante": {
        "menu_price_n": None,
        "importe_linea": None
    },
    "hora_inicio_invalida": {
        "hora_inicio": None
    },
    "hora_fin_invalida": {
        "hora_fin": None
    }
}

columnas = list(base_esperada.keys())

filas = (
    spark.table(
        "workspace.taco_bueno_pruebas.silver_pos_lines"
    )
    .select("caso_prueba", *columnas)
    .collect()
)

# Comprobar que ningún caso desapareció o se duplicó.
nombres = [fila["caso_prueba"] for fila in filas]

assert len(filas) == 8, "La tabla debe conservar ocho filas."
assert set(nombres) == set(casos_esperados), (
    "Los casos encontrados no coinciden con los esperados."
)

fallos = []

for fila in filas:
    actual = fila.asDict()
    nombre = actual["caso_prueba"]

    esperado = base_esperada.copy()
    esperado.update(casos_esperados[nombre])

    # Comparar únicamente la hora de los timestamps.
    for columna in ["hora_inicio", "hora_fin"]:
        if actual[columna] is not None:
            actual[columna] = actual[columna].time()

    diferencias = [
        columna
        for columna in columnas
        if actual[columna] != esperado[columna]
    ]

    if diferencias:
        detalle = f"{nombre}: {', '.join(diferencias)}"
        fallos.append(detalle)
        print("ERROR:", detalle)
    else:
        print("CORRECTO:", nombre)

if fallos:
    raise AssertionError(
        "Fallaron las pruebas: " + "; ".join(fallos)
    )

print(
    "\nPRUEBA APROBADA: los ocho casos produjeron "
    "los resultados esperados."
)